In [ ]:
using LinearAlgebra
using Plots

function construir_matriz_FTCS(N::Int, dx::Float64, dt::Float64, a::Float64, mu::Float64)
    r = mu * dt / (dx^2)
    sigma = a * dt / dx

    alpha = r - sigma / 2.0  # Subdiagonal
    beta  = 1.0 - 2.0 * r    # Diagonal principal
    gamma = r + sigma / 2.0  # Superdiagonal

    # Matriz Tridiagonal de Toeplitz
    dl = fill(alpha, N - 1)
    d  = fill(beta, N)
    du = fill(gamma, N - 1)

    A = Tridiagonal(dl, d, du)
    return A, r, sigma
end

# ------------------------------------------------------------------------------
# 2. Función para simular el esquema en el tiempo
# ------------------------------------------------------------------------------
function simular(A, U0, pasos::Int)
    N = length(U0)
    historia = zeros(N, pasos + 1)
    historia[:, 1] = U0
    
    U = copy(U0)
    for n in 1:pasos
        U = A * U
        historia[:, n+1] = U
    end
    return historia
end

# ------------------------------------------------------------------------------
# 3. Configuración del Dominio y Condición Inicial
# ------------------------------------------------------------------------------
N = 100
L = 1.0
dx = L / (N + 1)
x = range(dx, L - dx, length=N)

# Condición inicial: Pulso gaussiano en el centro
U0 = exp.(-100.0 .* (x .- 0.5).^2);

In [ ]:
# ==============================================================================
# Convección-Difusión ESTABLE
# ==============================================================================
a = 1.0
mu = 0.05

# Cálculo de dt límite teórico
dt_difusion = (dx^2) / (2 * mu)
dt_conveccion = (2 * mu) / (a^2)
dt_max = min(dt_difusion, dt_conveccion)

dt_estable = 0.8 * dt_max  # Elegimos un dt un 20% menor al límite
A_est, r_est, sigma_est = construir_matriz_FTCS(N, dx, dt_estable, a, mu)

# Radio espectral
eigs_est = eigvals(A_est)
rho_est = maximum(abs.(eigs_est))

println("=== CASO ESTABLE ===")
println("r = $r_est, sigma = $sigma_est")
println("dt elegido: $dt_estable (dt_max = $dt_max)")
println("Radio espectral rho(A): $rho_est (<= 1 asegura estabilidad)\n")

# Simulación
pasos = 200
hist_est = simular(A_est, U0, pasos)

p1 = plot(x, hist_est[:, 1], label="t = 0", lw=2, title="Caso Estable (rho = $(round(rho_est, digits=4)))")
plot!(p1, x, hist_est[:, 50], label="t = 50 dt", lw=2)
plot!(p1, x, hist_est[:, end], label="t = 200 dt", lw=2, xlabel="x", ylabel="U(x)")

In [ ]:
# ==============================================================================
# Convección-Difusión INESTABLE (dt demasiado grande)
# ==============================================================================
dt_inestable = 1.2 * dt_max  # Supera el límite de estabilidad
A_inest, r_inest, sigma_inest = construir_matriz_FTCS(N, dx, dt_inestable, a, mu)

eigs_inest = eigvals(A_inest)
rho_inest = maximum(abs.(eigs_inest))

println("=== CASO INESTABLE ===")
println("dt elegido: $dt_inestable (dt_max = $dt_max)")
println("Radio espectral rho(A): $rho_inest (> 1 causa explosión numérica)\n")

hist_inest = simular(A_inest, U0, 80)

p2 = plot(x, hist_inest[:, 1], label="t = 0", lw=2, title="Caso Inestable (rho = $(round(rho_inest, digits=4)))")
plot!(p2, x, hist_inest[:, 40], label="t = 40 dt", lw=2)
plot!(p2, x, hist_inest[:, 80], label="t = 80 dt", lw=2, xlabel="x", ylabel="U(x)")


In [ ]:
# ==============================================================================
# CASO LÍMITE a = 0 (Difusión Pura)
# ==============================================================================
a_dif = 0.0
mu_dif = 0.05
dt_dif = 0.9 * ((dx^2) / (2 * mu_dif)) # Cumple dt <= dx^2 / (2*mu)

A_dif, r_dif, sigma_dif = construir_matriz_FTCS(N, dx, dt_dif, a_dif, mu_dif)
rho_dif = maximum(abs.(eigvals(A_dif)))

println("=== DIFUSIÓN PURA (a = 0) ===")
println("Radio espectral rho(A): $rho_dif (Estable)\n")

=== EXPERIMENTO 3: DIFUSIÓN PURA (a = 0) ===
Radio espectral rho(A): 0.9995646540627894 (Estable)



In [ ]:
# ==============================================================================
# CASO LÍMITE mu = 0 (Convección Pura - Inestabilidad Incondicional)
# ==============================================================================
a_conv = 1.0
mu_conv = 0.0
dt_conv = 0.0001 # Cualquier dt positivo

A_conv, r_conv, sigma_conv = construir_matriz_FTCS(N, dx, dt_conv, a_conv, mu_conv)
rho_conv = maximum(abs.(eigvals(A_conv)))

println("=== EXPERIMENTO 4: CONVECCIÓN PURA (mu = 0) ===")
println("dt = $dt_conv")
println("Radio espectral rho(A): $rho_conv (> 1 incondicionalmente)")

# Mostrar gráficos
plot(p1, p2, layout=(2, 1), size=(800, 600))